Small Example to test correctness with N=4, M=2 and an anisotropic number of blocks.

In [ ]:
%%writefile mat_calc.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <iostream>

#define N 4
#define M 2
#define THREADS 4

__global__ void matmult(float *a, float *b, float *x, float *c, int n, int m) {
  int row = threadIdx.x + blockIdx.x * blockDim.x;
  int column = threadIdx.y + blockIdx.y * blockDim.y;

  int global_id = row * m + column;
  int thread_id = global_id;
  while (thread_id < n*n) {
    int i = thread_id / n;
    float sum = 0;
    for (int j= 0; j<m; j++) {
      sum += a[i * m + j] * x[i + j * n];
    }

    c[thread_id] = sum + b[i];
    thread_id += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++) {
      a[j+i*M] = i < N/2 ;
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N*N;i++) {
    b[i] = 0;
  }
}

void fill_x(float *x){
  for(int i=0;i<M;i++) {
    for(int j=0;j<N;j++) {
      x[j+i*N] = i<M/2;
    }
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*M*sizeof(float);
  const int output_size = N*N*sizeof(float);

  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, output_size);
	cudaMalloc((void**)&d_c, output_size);
  cudaMalloc((void**)&d_x, mat_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(output_size);
	c = (float*)malloc(output_size);
  x = (float*)malloc(mat_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, output_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, mat_size, cudaMemcpyHostToDevice);

  dim3 blocks(2, 4);
  dim3 threads(2, 2);

  matmult << <blocks, threads >> >(d_a, d_b, d_x, d_c, N, M);

  cudaMemcpy(c, d_c, output_size, cudaMemcpyDeviceToHost);

  std::cout << "A: " << std::endl;
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++){
      std::cout << a[i*M + j] << " ";
    }
    std::cout << std::endl;
  }

  std::cout << "X: " << std::endl;
  for(int i=0; i<M; i++) {
    for(int j=0; j<N; j++){
      std::cout << x[i*N + j] << " ";
    }
    std::cout << std::endl;
  }

  std::cout << "B: " << std::endl;
  for(int i=0; i<N; i++) {
    for(int j=0; j<N; j++){
      std::cout << b[i*N + j] << " ";
    }
    std::cout << std::endl;
  }


  std::cout << "C: " << std::endl;
  for(int i=0; i<N; i++) {
    for(int j=0; j<N; j++){
      std::cout << c[i*N + j] << " ";
    }
    std::cout << std::endl;
  }

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_x);
  cudaFree(d_c);

  return 0;

}



Writing mat_calc.cu


In [ ]:
!nvcc -arch=sm_75 mat_calc.cu -o mat_calc
!./mat_calc

A: 
1 1 
1 1 
0 0 
0 0 
X: 
1 1 1 1 
0 0 0 0 
B: 
0 0 0 0 
0 0 0 0 
0 0 0 0 
0 0 0 0 
C: 
1 1 1 1 
1 1 1 1 
0 0 0 0 
0 0 0 0 


Example to measure time between single and multi core

In [ ]:
%%writefile mat_calc_timed.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <iostream>
#include <chrono>

#define N 8192 / 2
#define M 2048 / 2
#define THREADS 256

__global__ void matmult(float *a, float *b, float *x, float *c, int n, int m) {
  int row = threadIdx.x + blockIdx.x * blockDim.x;
  int column = threadIdx.y + blockIdx.y * blockDim.y;

  int global_id = row * m + column;
  int thread_id = global_id;
  while (thread_id < n*n) {
    int i = thread_id / n;
    float sum = 0;
    for (int j= 0; j<m; j++) {
      sum += a[i * m + j] * x[i + j * n];
    }

    c[thread_id] = sum + b[i];
    thread_id += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++) {
      a[j+i*M] = i < N/2 ;
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N*N;i++) {
    b[i] = 0;
  }
}

void fill_x(float *x){
  for(int i=0;i<M;i++) {
    for(int j=0;j<N;j++) {
      x[j+i*N] = i<M/2;
    }
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*M*sizeof(float);
  const int output_size = N*N*sizeof(float);

  std::chrono::steady_clock::time_point start_startup = std::chrono::steady_clock::now();
  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, output_size);
	cudaMalloc((void**)&d_c, output_size);
  cudaMalloc((void**)&d_x, mat_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(output_size);
	c = (float*)malloc(output_size);
  x = (float*)malloc(mat_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, output_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, mat_size, cudaMemcpyHostToDevice);

  std::chrono::steady_clock::time_point stop_startup = std::chrono::steady_clock::now();

  // Single Core
  dim3 blocks_single(1, 1);
  dim3 threads_single(1, 1);
  std::chrono::steady_clock::time_point start_single = std::chrono::steady_clock::now();
  matmult << <blocks_single, threads_single >> >(d_a, d_b, d_x, d_c, N, M);
  std::chrono::steady_clock::time_point stop_single = std::chrono::steady_clock::now();

  // Multi Core
  dim3 blocks_multi((N / THREADS) / 2, (N / THREADS) / 2);
  dim3 threads_multi(THREADS, THREADS);
  std::chrono::steady_clock::time_point start_multi = std::chrono::steady_clock::now();
  matmult << <blocks_multi, threads_multi >> >(d_a, d_b, d_x, d_c, N, M);
  std::chrono::steady_clock::time_point stop_multi = std::chrono::steady_clock::now();

  cudaMemcpy(c, d_c, output_size, cudaMemcpyDeviceToHost);

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_x);
  cudaFree(d_c);

  auto startup_time = std::chrono::duration_cast<std::chrono::microseconds>(stop_startup - start_startup).count();
  auto single_time = std::chrono::duration_cast<std::chrono::microseconds>(stop_single - start_single).count();
  auto multi_time = std::chrono::duration_cast<std::chrono::microseconds>(stop_multi - start_multi).count();

  std::cout << "Start Up: " <<  startup_time << std::endl;
  std::cout << "Single Core: " <<  single_time << std::endl;
  std::cout << "Multi Core: " << multi_time << std::endl;
  if (multi_time > 0){
    std::cout << "Speedup: " << single_time / multi_time << std::endl;
  }
  return 0;

}



Overwriting mat_calc_timed.cu


In [ ]:
!nvcc -arch=sm_75 mat_calc_timed.cu -o mat_calc_timed
!./mat_calc_timed

Start Up: 421899
Single Core: 220
Multi Core: 1
Speedup: 220


| | Startup Time | Single Time | Multi Time
| --- | --- | ---- | ----
|/8 | 294655 | 130 | 0
|/4 | 292759 | 173 | 1
|/2 | 354014 | 214 | 1
|/1 | 423875 | 220 | 1

Measure Time for Number of Threads

In [ ]:
%%writefile mat_calc_threads.cu

#include "cuda_runtime.h"
#include <stdlib.h>
#include <stdio.h>
#include <iostream>
#include <chrono>

#define N 8192 / 2
#define M 2048 / 2
#define THREADS 512

__global__ void matmult(float *a, float *b, float *x, float *c, int n, int m) {
  int row = threadIdx.x + blockIdx.x * blockDim.x;
  int column = threadIdx.y + blockIdx.y * blockDim.y;

  int global_id = row * m + column;
  int thread_id = global_id;
  while (thread_id < n*n) {
    int i = thread_id / n;
    float sum = 0;
    for (int j= 0; j<m; j++) {
      sum += a[i * m + j] * x[i + j * n];
    }

    c[thread_id] = sum + b[i];
    thread_id += blockDim.x * gridDim.x;
  }
}

void fill_a(float *a){
  for(int i=0; i<N; i++) {
    for(int j=0; j<M; j++) {
      a[j+i*M] = i < N/2 ;
    }
  }

}

void fill_b(float *b){
  for(int i=0;i<N*N;i++) {
    b[i] = 0;
  }
}

void fill_x(float *x){
  for(int i=0;i<M;i++) {
    for(int j=0;j<N;j++) {
      x[j+i*N] = i<M/2;
    }
  }
}

int main() {
  float *a, *b, *c, *x;
  float *d_a, *d_b, *d_c, *d_x;

  const int mat_size = N*M*sizeof(float);
  const int output_size = N*N*sizeof(float);

  std::chrono::steady_clock::time_point start_startup = std::chrono::steady_clock::now();
  cudaMalloc((void**)&d_a, mat_size);
	cudaMalloc((void**)&d_b, output_size);
	cudaMalloc((void**)&d_c, output_size);
  cudaMalloc((void**)&d_x, mat_size);

  a = (float*)malloc(mat_size);
	b = (float*)malloc(output_size);
	c = (float*)malloc(output_size);
  x = (float*)malloc(mat_size);

  fill_a(a);
  fill_b(b);
  fill_x(x);

  cudaMemcpy(d_a, a, mat_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, b, output_size, cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, x, mat_size, cudaMemcpyHostToDevice);

  std::chrono::steady_clock::time_point stop_startup = std::chrono::steady_clock::now();

  auto startup_time = std::chrono::duration_cast<std::chrono::microseconds>(stop_startup - start_startup).count();
  std::cout << "Start Up: " <<  startup_time << std::endl;

  for (int div = 4; div > -1; div--) {
    // Multi Core
    int threads = THREADS / std::pow(2, div);

    dim3 blocks_multi((N / threads) / 2, (N / threads) / 2);
    dim3 threads_multi(threads, threads);
    std::chrono::steady_clock::time_point start_multi = std::chrono::steady_clock::now();
    matmult << <blocks_multi, threads_multi >> >(d_a, d_b, d_x, d_c, N, M);
    std::chrono::steady_clock::time_point stop_multi = std::chrono::steady_clock::now();

    auto multi_time = std::chrono::duration_cast<std::chrono::microseconds>(stop_multi - start_multi).count();
    std::cout <<  "Threads: " << threads << std::endl;
    std::cout << "Multi Core: " << multi_time << std::endl;
    std::cout << std::endl;
  }

  // cudaMemcpy(c, d_c, output_size, cudaMemcpyDeviceToHost);
  std::chrono::steady_clock::time_point start_freeup = std::chrono::steady_clock::now();

  free(a);
  free(b);
  free(x);
  free(c);

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_x);
  cudaFree(d_c);

  std::chrono::steady_clock::time_point stop_freeup = std::chrono::steady_clock::now();

  auto freeup_time = std::chrono::duration_cast<std::chrono::microseconds>(stop_freeup - start_freeup).count();
  std::cout << "Freeup Time: " << freeup_time << std::endl;

  return 0;

}



Overwriting mat_calc_threads.cu


In [ ]:
!nvcc -arch=sm_75 mat_calc_threads.cu -o mat_calc_threads
!./mat_calc_threads

Start Up: 430807
Threads: 32
Multi Core: 199

Threads: 64
Multi Core: 2

Threads: 128
Multi Core: 0

Threads: 256
Multi Core: 0

Threads: 512
Multi Core: 0

Freeup Time: 467781496
